# Axis 5 — Memory Hierarchy & Expert Placement (Windows iGPU / PyTorch)

**Question (team page).** How does placing experts across memory tiers affect end-to-end latency and throughput?

**Tiers.** The iGPU machine has **three logical tiers but only two physical memories**: **shared GPU memory** (the bank the GPU computes from), **system RAM** (the RAM tier, one memory-to-memory copy away) and **disk** (NVMe). The first two are the same DRAM, so a RAM-tier hit saves the disk read but not the copy, and the RAM tier takes memory the bank could have used. The Mac has two tiers (unified memory, disk); the dGPU has three physically separate ones (VRAM, RAM over PCIe, disk). The Mac question — *which experts should live in the fast tier, fixed or changing?* — is asked here for the GPU tier, **with and without a RAM tier** underneath.

## Strategies for the GPU tier, at the same budget B (experts in shared GPU memory)
| Strategy | GPU tier | Changes over time? |
|---|---|---|
| **static** | the B − 64 experts used most in a *profiling* run, pinned for good, plus a 64-slot staging area (the minimum to compute any layer call) | almost not at all |
| **hybrid** | B/2 pinned hottest experts + B/2 LRU slots | half |
| **dynamic** | B LRU slots, nothing pinned | fully |

## The RAM tier
| Layout | A GPU-tier miss is served from |
|---|---|
| **three tiers** (`ram_tier` = R) | the RAM tier if it holds the expert (one host → GPU-memory copy), else disk → RAM tier → GPU memory. The RAM tier is R buffers managed by LRU; it never holds the full bank (R < 1,024). On the iGPU it is ordinary (pageable) RAM, sized to what is left after the largest timed bank. |
| **two tiers** (`ram_tier` = 0) | disk → pinned staging buffer → GPU memory, as on the Mac |

* Profiling uses a **different** set of prompts from the evaluation, so pinning is not tuned to the test.
* Pinned GPU-tier experts are loaded before the run (startup cost reported separately, not counted as misses).
* Every disk read is unbuffered, so the Windows file cache is not a hidden fourth tier.
* `runs` holds the three-tier runs (the headline result on Windows, used by the cross-device figures); `runs_two_tier` holds the same runs without the RAM tier.

**Hypothesis.** A hierarchy that keeps frequently reused experts in the fastest tier, stages warm experts in an intermediate tier, and fetches only cold tails from disk beats a single uniform policy. `MOE_SMOKE=1` runs a tiny configuration.

In [ ]:
import os, sys, time, json, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Shared code: the `moe` package in src/. `pip install -e ".[windows]"` installs it; the path
# line below makes the notebook work without installing.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'moe').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moe import common, torch_cache as tc, results as rs, figures as fg   # tc is Windows (PyTorch)
for m in (common, tc, rs, fg):
    importlib.reload(m)                # pick up edits to src/moe without restarting the kernel

# MOE_SMOKE=1 runs a tiny configuration end to end (used to test the notebook).
SMOKE = os.environ.get('MOE_SMOKE') == '1'
AXIS_DIR = ROOT / 'experiments' / 'axis-5-memory-tiering'
RESULTS = (ROOT / 'experiments' / '.cache' / 'smoke' / 'axis-5-memory-tiering') if SMOKE else (AXIS_DIR / 'results')
RESULTS.mkdir(parents=True, exist_ok=True)

DEVICE = 'windows-igpu'                 # results file: <axis>/results/windows-igpu.json
AXIS = 'axis-5-memory-tiering'
TABLES = {}                             # every table this run produces, saved at the end

GPU = tc.default_device('xpu')          # the Intel integrated GPU (Windows + XPU only); K counts experts in
                                        # its shared GPU memory (system RAM the GPU may use)
torch.set_grad_enabled(False)

plt.rcParams.update({'figure.figsize': (11, 5), 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})
print('SMOKE run' if SMOKE else 'Full run', '| results ->', RESULTS.relative_to(ROOT))
print(tc.system_info(GPU))

## Configuration

In [ ]:
BUDGETS = [192, 384, 576]          # experts resident in shared GPU memory (bank = B x 12.6 MB)
RAM_TIER = 256                     # experts in the RAM tier (3.2 GB) if they fit; 0 = two tiers only
RAM_RESERVE_GB = 2.5               # RAM kept free for Windows and Python next to the bank and RAM tier
STAGING = 64                       # dynamic slots kept by the static strategy
HYBRID_PIN_FRACTION = 0.5
PROFILE_PER_DOMAIN = 2             # profiling prompts per kind (disjoint from evaluation; max 2+2)
EVAL_PER_DOMAIN = 2
MAX_NEW_TOKENS = 48
REPLAY_BUDGETS = [128, 192, 256, 384, 512, 576, 768]   # denser curve by exact replay

if SMOKE:
    BUDGETS, PROFILE_PER_DOMAIN, EVAL_PER_DOMAIN, MAX_NEW_TOKENS = [192], 1, 1, 6
    REPLAY_BUDGETS, RAM_TIER = [128, 192, 256], 64
print('GPU budgets:', BUDGETS, '| RAM tier:', RAM_TIER, 'experts | strategies: static, hybrid, dynamic')
CONFIG = rs.config_from(globals())   # recorded in the results file

## Model (non-expert weights only) and correctness gate

In [ ]:
MODEL_DIR = tc.default_model_dir()       # allenai/OLMoE-1B-7B-0125-Instruct (BF16), downloaded beforehand
assert list(MODEL_DIR.glob('model*.safetensors')),     f'Checkpoint not found in {MODEL_DIR}: download allenai/OLMoE-1B-7B-0125-Instruct there first.'
model, tokenizer, index = tc.load_model(MODEL_DIR, device=GPU)
print(index.describe())
print(f'Non-expert weights resident on {GPU}: {tc.non_expert_bytes(model) / 1e9:.2f} GB '
      f'| expert reads: {tc.ExpertReader(index).mode}')
# Measured ONCE, before any expert bank exists: free memory read later in the run is unreliable.
# On the iGPU the bank is system RAM, so K must fit in the shared GPU memory (half of RAM)
# AND in the RAM free right now (see tc.max_safe_capacity).
SAFE_K = tc.max_safe_capacity(index, GPU)
print(f'Largest K that fits in shared GPU memory and free RAM right now: {SAFE_K} '
      f'(bank {SAFE_K * index.expert_bytes / 1e9:.1f} GB)')
print('Memory now:', {k: round(v, 2) for k, v in tc.memory_snapshot(GPU).items()})

# Correctness gate: one layer through the K-slot bank vs the stock transformers layer
# (all 64 experts of that layer resident, eager expert loop). Must be bit-identical.
diff = tc.check_layer_exactness(model, index, capacity=64)
assert diff == 0.0, diff
print(f'Layer exactness vs stock transformers: max |diff| = {diff}')

In [ ]:
# Budgets whose bank fits (SAFE_K was measured at load, before any bank existed).
timed = [b for b in BUDGETS if b <= SAFE_K]
assert timed, f'No budget fits in free shared GPU memory / RAM (largest safe K = {SAFE_K}). Close other programs and re-run.'
# On the iGPU the bank and the RAM tier come out of the same 16 GB of RAM, so the RAM tier gets
# only what is left once the largest timed bank is allocated. It is ordinary (pageable) memory:
# PyTorch's pinned allocator would round a multi-GB block up to the next power of two.
BANK_GB = max(timed) * index.expert_bytes / 1e9
SAFE_RAM = tc.max_host_capacity(index, reserve_gb=RAM_RESERVE_GB + BANK_GB)
R = min(RAM_TIER, SAFE_RAM)
if R < RAM_TIER:
    print(f'RAM tier reduced from {RAM_TIER} to {R} experts to fit next to a {BANK_GB:.1f} GB bank.')
host = tc.HostTier(index, R, device=GPU) if R else None    # allocated once, emptied before every run
if host is not None:
    print(f'RAM tier: {R} experts = {host.bytes / 1e9:.1f} GB | page-locked: {host.pinned} (pageable on the iGPU) '
          f'(allocated in {host.alloc_s:.1f}s)')

## Profiling run: which experts are hot?

In [ ]:
# Per kind of request, the first PROFILE_PER_DOMAIN prompts are used to find hot experts and the
# following EVAL_PER_DOMAIN are the held-out evaluation traffic (same split as the Mac).
prompt_file = rs.load_prompts(AXIS_DIR)   # axis-5-memory-tiering/prompts.json, shared by every device
PROMPTS = prompt_file['PROMPTS']
profile_reqs = tc.requests(PROMPTS, n_per_domain=PROFILE_PER_DOMAIN, order='interleave')
eval_reqs = tc.requests(PROMPTS, n_per_domain=EVAL_PER_DOMAIN, order='interleave', offset=PROFILE_PER_DOMAIN)
assert not {p['text'] for p in profile_reqs} & {p['text'] for p in eval_reqs}

cache = tc.ExpertCache(index, min(256, SAFE_K), 'LRU', device=GPU)   # outputs do not depend on K
_, rec = tc.run_workload(model, tokenizer, cache, profile_reqs, MAX_NEW_TOKENS, log=None)
cache.close(); tc.detach_cache(model)
prof = tc.events_frame(rec.events)
ranking = prof.groupby(['layer', 'eid']).size().sort_values(ascending=False)
hot = [tuple(map(int, k)) for k in ranking.index]
print(f'profiling: {len(profile_reqs)} requests, {len(prof):,} accesses, {len(hot)} distinct experts')
print('top-10 experts (layer, id):', hot[:10])
print('share of accesses covered by the top 128 / 256 / 512:',
      [f'{ranking.iloc[:n].sum() / ranking.sum():.0%}' for n in (128, 256, 512)])

## Real runs on the evaluation requests
Every GPU budget that fits × every strategy × {three tiers, two tiers}. Each run starts with an empty GPU cache and an empty RAM tier. Exact replay of both tiers is asserted against the real counters.

In [ ]:
def make_strategy(name, B):
    if name == 'dynamic':
        return tc.make_policy('LRU', B), []
    n_pin = B - STAGING if name == 'static' else int(B * HYBRID_PIN_FRACTION)
    pins = hot[:n_pin]
    return tc.TieredPolicy(B, pins, 'LRU'), pins

if len(timed) < len(BUDGETS):
    print(f'Not timed (bank would not fit in shared GPU memory / free RAM): '
          f'{[b for b in BUDGETS if b > SAFE_K]} -> see the replay curve below')
tiers = [R, 0] if R else [0]
rows, traces = [], {}
for B in timed:
    for ram in tiers:
        for strat in ['static', 'hybrid', 'dynamic']:
            tc.free_memory(); tc.reset_peak_memory(GPU)
            pol, pins = make_strategy(strat, B)
            if ram:
                host.reset()
            cache = tc.ExpertCache(index, B, pol, device=GPU, host_tier=host if ram else None)
            startup_ms = cache.preload(pins)
            t0 = time.perf_counter()
            df, rec = tc.run_workload(model, tokenizer, cache, eval_reqs, MAX_NEW_TOKENS, log=None)
            s = cache.summary()
            p2, _ = make_strategy(strat, B)
            if ram:
                hits, ram_hits = tc.replay_two_tier(rec.events, p2, ram)
                assert int(ram_hits.sum()) == s['ram_hits'], 'RAM-tier replay mismatch'
            else:
                hits = tc.replay(rec.events, p2)
            assert int(hits.sum()) == s['hits'], 'replay mismatch'
            dec_tok = (df.new_tokens - 1).sum()
            acc = tc.events_frame(rec.events)
            rows.append({'B': B, 'strategy': strat, 'ram_tier': ram, 'pinned': len(pins), 'hit_rate': s['hit_rate'],
                         'decode_hit_rate': hits[(acc.phase == 'decode').to_numpy()].mean(),
                         'ram_hit_rate': s.get('ram_hit_rate', float('nan')),
                         'GB_loaded': s['GB_loaded'], 'disk_GB': s['disk_GB'], 'io_s': s['io_ms'] / 1000,
                         'disk_s': s['disk_s'], 'h2d_s': s['h2d_s'],
                         'startup_s': startup_ms / 1000, 'startup_GB': len(pins) * index.expert_bytes / 1e9,
                         'decode_tok_s': dec_tok / df.decode_s.sum(), 'ttft_mean_s': df.ttft_s.mean(),
                         'wall_s': time.perf_counter() - t0, 'peak_GB': tc.peak_memory_gb(GPU)})
            traces[(B, ram, strat)] = rec.events
            cache.close(); tc.detach_cache(model)
            r = rows[-1]
            print(f'B={B} RAM={ram:4d} {strat:8s}: GPU hit {r["hit_rate"]:.1%} (decode {r["decode_hit_rate"]:.1%}) | '
                  f'RAM hit {r["ram_hit_rate"]:.1%} | {r["disk_GB"]:.1f} GB from disk | {r["decode_tok_s"]:.2f} tok/s | '
                  f'TTFT {r["ttft_mean_s"]:.2f}s | startup {r["startup_s"]:.1f}s')
res = pd.DataFrame(rows)
TABLES['runs_all'] = res
TABLES['runs'] = res[res.ram_tier == tiers[0]].reset_index(drop=True)          # headline: three tiers
if R:
    TABLES['runs_two_tier'] = res[res.ram_tier == 0].reset_index(drop=True)
res.style.format(precision=2)

## Denser budget curve by exact replay
The evaluation access stream is the same for every strategy (outputs do not depend on the cache), so each strategy can be replayed at more budgets — including ones too large to time on this GPU — with and without the RAM tier. `GB_read` is what comes from disk (with the RAM tier when there is one); `GB_to_vram` is every GPU-tier miss (the column keeps the dGPU's name so the two Windows files line up).

In [ ]:
events = next(iter(traces.values()))
acc = tc.events_frame(events)
dec = (acc.phase == 'decode').to_numpy()
rep = []
for B in REPLAY_BUDGETS:
    for strat in ['static', 'hybrid', 'dynamic']:
        if strat == 'static' and B <= STAGING:
            continue
        pol, _ = make_strategy(strat, B)
        if R:
            h, rh = tc.replay_two_tier(events, pol, R)
            from_disk = int((~rh).sum())
        else:
            h = tc.replay(events, pol)
            rh, from_disk = np.array([]), int((~h).sum())
        rep.append({'B': B, 'strategy': strat, 'hit_rate': h.mean(), 'decode_hit_rate': h[dec].mean(),
                    'ram_tier': R, 'ram_hit_rate': rh.mean() if len(rh) else float('nan'),
                    'GB_read': from_disk * index.expert_bytes / 1e9,
                    'GB_to_vram': (~h).sum() * index.expert_bytes / 1e9})
rep = pd.DataFrame(rep)
TABLES['replay'] = rep
rep.pivot(index='B', columns='strategy', values='hit_rate').style.format('{:.1%}').highlight_max(axis=1, color='#c6efce')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for i, (strat, d) in enumerate(rep.groupby('strategy')):
    axes[0].plot(d.B, d.hit_rate, '-o', color=f'C{i}', label=strat)
    axes[1].plot(d.B, d.GB_to_vram, '-o', color=f'C{i}', label=f'{strat}: two tiers (every GPU miss)')
    if R:
        axes[1].plot(d.B, d.GB_read, '--s', color=f'C{i}', label=f'{strat}: three tiers (RAM-tier misses)')
for i, (strat, d) in enumerate(res.groupby('strategy')):
    for ram, ls in ([(R, '-o'), (0, '--s')] if R else [(0, '-o')]):
        dd = d[d.ram_tier == ram]
        if len(dd):
            axes[2].plot(dd.B, dd.decode_tok_s, ls, color=f'C{i}', label=f'{strat}, RAM tier {ram}')
axes[0].set_title('GPU-tier hit rate (replay)'); axes[0].set_ylabel('hit rate')
axes[1].set_title('Bytes read from disk (replay)'); axes[1].set_ylabel('GB')
axes[2].set_title('Decode throughput (real runs)'); axes[2].set_ylabel('tokens / s')
for ax in axes:
    ax.set_xlabel('GPU budget B (experts)'); ax.legend(fontsize=7)
fig.tight_layout(); plt.show()

## Hypothesis check (descriptive)

In [ ]:
for B, d in rep.groupby('B'):
    d = d.set_index('strategy').hit_rate
    print(f'B={B:4d}: best = {d.idxmax():8s} ({d.max():.1%}); ' + ', '.join(f'{k} {v:.1%}' for k, v in d.items()))
for (B, ram), d in res.groupby(['B', 'ram_tier']):
    d = d.set_index('strategy')
    print(f'real B={B} RAM tier {ram}: fastest decode = {d.decode_tok_s.idxmax()} ({d.decode_tok_s.max():.2f} tok/s); '
          f'static pays {d.loc["static", "startup_s"]:.1f}s of startup loading ({d.loc["static", "startup_GB"]:.1f} GB)')
if R:
    both = res.pivot_table(index=['B', 'strategy'], columns='ram_tier', values='decode_tok_s')
    gain = both[R] / both[0] - 1
    print(f'\nRAM tier of {R} experts: decode throughput change vs two tiers: '
          + ', '.join(f'B={b} {s} {g:+.0%}' for (b, s), g in gain.items()))
print('\nA tiered layout "wins" only if pinning what was hot in profiling beats pure recency on new prompts;'
      ' the RAM tier helps when the GPU misses it catches cost less than a disk read; on the iGPU it also takes'
      ' RAM the bank could have used.')

## Notes for reading the results

* **Within-device trends first.** These are PyTorch / BF16 / XPU numbers from an Intel Arc integrated GPU (Core Ultra 5 125H, 16 GB). The Mac runs MLX / F16 on unified memory: compare the *shape* of the curves across devices, not absolute values as one controlled experiment. Like the Mac, this GPU computes from system RAM; unlike the Mac, a miss still pays a copy from the staging buffer into the GPU's allocation (`h2d_s`), a memory-to-memory copy rather than a PCIe transfer as on the dGPU.
* **No page cache to purge.** Expert reads are unbuffered (`reads` = `windows-unbuffered` in the system info), so every miss really goes to the disk. If the reader had to fall back to buffered reads it prints a warning and the mode says `buffered`.
* **K is the shared-GPU-memory tier.** This GPU has no memory of its own (a 128 MB carve-out): the bank lives in system RAM that Windows lets the GPU use, at most half of RAM. The timed K values are those whose bank fits in that *and* in the RAM free at load (`SAFE_K`); larger K are replay-only (hit rates exact, no timing). Close other programs, browsers especially, before a full run: on this machine they use the same memory as the bank.

* **RAM tier is pageable here.** On the dGPU the RAM tier is page-locked; on the iGPU it is ordinary RAM (`page-locked: False`), sized by `SAFE_RAM` to what is left after the largest timed bank and `RAM_RESERVE_GB`. Watch Task Manager during a full run: if Windows starts paging, close programs or lower `RAM_TIER`.

## Save results
Every table above goes into **one file for this device**: `results/windows-igpu.json`, with the configuration, prompts and machine it ran on.
The Mac and Windows dGPU notebooks write `mac.json` / `windows-dgpu.json` next to it in the same format (see `experiments/README.md`).

In [ ]:
CONFIG['SAFE_K'] = globals().get('SAFE_K')
CONFIG['RAM_TIER_USED'] = R
path = rs.save(RESULTS, AXIS, DEVICE, TABLES, config=CONFIG, prompts=rs.load_prompts(AXIS_DIR),
               system=tc.system_info(GPU))
print('Saved', path.relative_to(ROOT), '| tables:', {k: len(v) for k, v in TABLES.items()})

## Cross-device figures
The report figures, drawn from every device's results file in this folder (colour = device). Only shown here;
`python -m moe.figures` writes them to `figures/` once all devices have results.

In [ ]:
figs = fg.make(AXIS, RESULTS)